In [110]:
import pandas as pd
import requests
import numpy as np
from datetime import datetime, timedelta
from bcb import currency
pd.set_option('future.no_silent_downcasting', True)

In [111]:
def processar_dados_vendas_td(url_sales, url_td, caminho_saida_csv):
    """
    Processa dados de estoque e vendas, retornando um DataFrame final consolidado e salvando-o como CSV.
    
    Args:
        url_sales (str): Link compartilhável da planilha de vendas (Google Sheets).
        url_inventory (str): Link compartilhável da planilha de trocas e devoluções (Google Sheets).
        caminho_saida_csv (str): Caminho completo para salvar o arquivo CSV resultante.

    Returns:
        pd.DataFrame: DataFrame consolidado com os dados processados.
    """
    # Carrega os dados das planilhas
    df_vendas = pd.read_csv(url_sales)
    df_td = pd.read_csv(url_td)
    
    # Exclui estoque WiiO Dropshipping e Unknown
    df_vendas = df_vendas.loc[df_vendas['Local'].isin(['Nacional', 'Internacional'])]

    # Altera o tipo de dado das colunas de data para datetime
    df_vendas['Data'] = pd.to_datetime(df_vendas['Data'], dayfirst=True)
    df_td['Data'] = pd.to_datetime(df_td['Data'], dayfirst=True)

    # Filtra os dataframes para que contenham apenas datas do dia 01-nov-2024 em diante
    data_corte = pd.to_datetime('2024-11-01')
    df_vendas = df_vendas[df_vendas['Data'] >= data_corte]
    df_td = df_td[df_td['Data'] >= data_corte]

    # Modificando os valores da coluna "Produto" com base na coluna "Variante"

    df_vendas.loc[df_vendas['Variante'] == "Compre 3 - Leve 5", "Produto"] = 'Kit 5 Cuecas'
    df_vendas.loc[df_vendas['Variante'] == "Compre 5 - Leve 10", "Produto"] = 'Kit 10 Cuecas'
       
    # Cria um dicionário para padronizar e uniformizar os nomes dos produtos
    
    mapeamento_nomes = {
        'Camisa Consolatio Ultra-Stretch':'Camisa Ultra-Stretch',
        'Camisa Consolatio X-Tretch':'Camisa X-Tretch',
        'Calça Social X-Tretch':'Calça X-Tretch',
        'Camisa Ultra-Stretch | Especial de Natal':'Camisa Ultra-Stretch',
        'Camisa X-Tretch | Especial de Natal':'Camisa X-Tretch',
        'Kit Cuecas Boxer Respiráveis (Compre 3, Leve 5)':'Kit 5 Cuecas',
        'Especial de Natal - Camisa X-Tretch':'Camisa X-Tretch',
        'OUTLET: Camisa Polo Ultra':'Camisa Polo Ultra','OUTLET: Calça Neo':'Calça Neo',
        'OUTLET: Camiseta TecModal':'Camiseta TecModal','OUTLET: Calça Social X-Tretch':'Calça X-Tretch',
        'OUTLET: Camisa X-Tretch':'Camisa X-Tretch','OUTLET: Camisa X-Tretch | Coleções Antigas':'Camisa X-Tretch',
        'OUTLET: Camisa X-Tretch Manga Curta':'Camisa X-Tretch Manga Curta',
        'OUTLET: Camisa Ultra-Stretch':'Camisa Ultra-Stretch',
        'Calça Consolatio X-Tretch':'Calça X-Tretch'
    }

    # Alterna os valores de acordo com o dicionário
    df_vendas['Produto'] = df_vendas['Produto'].replace(mapeamento_nomes)

    # Cria nova coluna 'PedidoSKU' para os dois dataframes para criar um indentificador único do produto dentro de cada pedido 
    # e possibilitar o merge
    df_vendas['PedidoSKU'] = df_vendas['Pedido'].astype(str) + '-' + df_vendas['SKU'].astype(str)
    df_td['PedidoSKU'] = '#' + df_td['Pedido'].astype(str) + '-' + df_td['SKU'].astype(str)

    # Une os dois dataframes a partir da coluna PedidoSKU
    df_vendas_td = pd.merge(df_vendas[['PedidoSKU','Pedido','Data','Produto','Cor','Tamanho','SKU','Quantidade','Faturamento bruto','Descontos', 'Faturamento líquido',
           'Frete', 'Frete preço', 'cidade', 'estado', 'Local']],
                            df_td[['Data','Status','Tipo','Motivo','Comentário do cliente','PedidoSKU']],
                            on='PedidoSKU',
                            how='left'
    )
    # Renomeia as colunas após merge
    df_vendas_td = df_vendas_td.rename(columns={'Data_x':'Data',
                                            'Data_y':'Data_TD',
                                            'Local':'Estoque',
                                            '#PedidoSKU':'ID',
                                            'Tipo':'T&D',
                                            'Motivo':'Motivo_T&D',
                                            'Comentário do cliente':'Comentário'
                                           })
    # Redefine o index do dataframe
    #df_vendas_td.set_index('ID',inplace=True)
    
    # Preenche valores nulos após o merge
    df_vendas_td['T&D']= df_vendas_td['T&D'].fillna('Sem T&D')
    df_vendas_td = df_vendas_td.fillna('')

    # Cria nova coluna com faturamento líquido corrigido para zerar faturamento de pedidos devolvidos
    #df_vendas_td['Faturamento_líquido_corrigido'] = np.where(
    #df_vendas_td['T&D'] == 'Devolução',
    #0,
    #df_vendas_td['Faturamento líquido']
    #)

    # Cria nova coluna contando quantos dias se passaram entre a data do pedido e da solicitação de T&D
    df_vendas_td['Dias_entre_datas'] = (df_vendas_td['Data_TD'] - df_vendas_td['Data']).dt.days 
    
    # Coloca as datas no formato brasileiro (DD/MM/YYYY)
    df_vendas_td['Data'] = df_vendas_td['Data'].dt.strftime('%d/%m/%Y')
    df_vendas_td['Data_TD'] = df_vendas_td['Data_TD'].dt.strftime('%d/%m/%Y')

    # Salva o novo dataframe como CSV
    df_vendas_td.to_csv(caminho_saida_csv,index=False,sep=';',decimal=',')

    return df_vendas_td

    

# USO DA FUNÇÃO:

df_vendas_td = processar_dados_vendas_td(
    url_sales="https://docs.google.com/spreadsheets/d/1u7TNCUlMPfIB2SsbjIbmJEnwpVuHo6FD_vQ-vvpCIIc/export?format=csv",
    url_td="https://docs.google.com/spreadsheets/d/1u7TNCUlMPfIB2SsbjIbmJEnwpVuHo6FD_vQ-vvpCIIc/export?format=csv&gid=1655161461",
    caminho_saida_csv=r"C:\Users\GabrielCielo\Desktop\consolatio\Databases\Vendas_TD_DB.csv"
)









In [112]:
# Função para obter a cotação do dólar na data desejada
def obter_cotacao(data):
    # Converter a data para o formato americano (MM-DD-YYYY)
    data_formatada = data.strftime("%m-%d-%Y")
    url = f"https://olinda.bcb.gov.br/olinda/servico/PTAX/versao/v1/odata/" \
          f"CotacaoDolarDia(dataCotacao=@dataCotacao)?@dataCotacao='{data_formatada}'&$format=json"
    try:
        response = requests.get(url)
        response.raise_for_status()
        dados = response.json()
        if len(dados['value']) > 0: 
            return dados['value'][0]['cotacaoCompra']
        else:
            return None  # Sem cotação para o dia
    except Exception as e:
        print(f"Erro ao obter cotação para {data}: {e}")
        return None

In [113]:
# Definir o período desejado
df_vendas_td['Data'] = pd.to_datetime(df_vendas_td['Data'],dayfirst=True)

data_inicio = df_vendas_td['Data'].min().strftime('%Y-%m-%d')
data_fim = datetime.today().strftime('%Y-%m-%d')

# Obter as cotações do dólar no período
df_cotacao = currency.get('USD', start=data_inicio, end=data_fim)

#df_cotacao['USD'] = df_cotacao['USD'].ffill().bfill()
# Resetar o índice para facilitar o merge
df_cotacao = df_cotacao.reset_index()

# Renomear a coluna 'date' para 'Data' para corresponder ao DataFrame de vendas
df_cotacao.rename(columns={'Date':'Data', 'USD':'Cotação_USD'},inplace=True)

# Converter a coluna 'Data' para datetime para garantir compatibilidade no merge

df_cotacao['Data'] = pd.to_datetime(df_cotacao['Data'])

# Fazer o merge dos DataFrames usando a coluna 'Data'
df_final = df_vendas_td.merge(df_cotacao, on='Data', how='left')


In [114]:
custos_predefinidos = {
    'Camisa X-Tretch': {1:15.77 , 2:25.57/2 , 3:35.06/3, 4:49.50/4, 5:60.05/5},
    'Camisa Ultra-Stretch': {1:18.86 , 2:32.61/2 , 3:46.95/3, 4:65.70/4, 5:84.05/5},
    'Camisa Polo Ultra': {1:12.85 , 2:21.49/2, 3:30.35/3, 4:38.67/4, 5:46.29/5},
    'Calça X-Tretch': {1:14.15, 2:23.97/2, 3:33.50/3, 4:41.50/4, 5:52.07/5},
    'Camisa X-Tretch Manga Curta': {1:18.5},
    'Kit 5 Cuecas': {1:11.1},
    'Kit 10 Cuecas': {1:17.95},
    'Necessaire de Viagem':{1:8.27}
}



In [115]:
def obter_custo_por_produto(produto, quantidade):
    if produto in custos_predefinidos:
        custos = custos_predefinidos[produto]
        # Encontrar a maior chave menor ou igual à quantidade
        max_quantidade = max((q for q in custos.keys() if q <= quantidade), default=None)
        if max_quantidade is not None:
            return custos[max_quantidade]
    return None  # Retorna None se não encontrar correspondência

In [116]:
df_final['Cotação_USD'] = df_final['Cotação_USD'].ffill().bfill()

df_final['Custo_Unitario_USD'] = df_final.apply(lambda row: obter_custo_por_produto(row['Produto'],row['Quantidade']),axis=1)

df_final['Custo_Unitario_BRL'] = df_final['Custo_Unitario_USD'] * df_final['Cotação_USD']

df_final.loc[df_final['Produto'] == 'Camiseta TecModal','Custo_Unitario_BRL'] = 36
df_final.loc[df_final['Produto'] == 'Calça Neo','Custo_Unitario_BRL'] = 121
df_final = df_final.fillna(0)

df_final['Custo_BRL'] = df_final['Custo_Unitario_BRL'] * df_final['Quantidade']

df_final['Faturamento líquido'] = df_final['Faturamento líquido'].str.replace(',','.')
df_final['Faturamento líquido'] = df_final['Faturamento líquido'].astype(float)

df_final['Lucro bruto'] = df_final['Faturamento líquido'] - df_final['Custo_BRL']

df_final['Margem de lucro'] = df_final['Lucro bruto'] / df_final['Faturamento líquido']

df_final['Margem de lucro'] = np.where(
    df_final['Faturamento líquido'] == 0,
    0, # Se receita for 0, margem de lucro percentual será 0%
    (df_final['Lucro bruto'] / df_final['Faturamento líquido']) 
)

In [117]:
df_final['Faturamento líquido'].info()

<class 'pandas.core.series.Series'>
RangeIndex: 24443 entries, 0 to 24442
Series name: Faturamento líquido
Non-Null Count  Dtype  
--------------  -----  
24443 non-null  float64
dtypes: float64(1)
memory usage: 191.1 KB


In [118]:
def determinar_estoque(grupo):
    if all(grupo == 'Nacional'):
        return 'Nacional'
    elif all(grupo == 'Internacional'):
        return 'Internacional'
    else:
        return 'Misto'

operacoes = {
    'Quantidade':'sum',
    'Faturamento líquido':'sum',
    'Custo_Unitario_USD':'sum',
    'Custo_Unitario_BRL':'sum',
    'Custo_BRL':'sum',
    'Lucro bruto':'sum',
    'Dias_entre_datas':'first',
    'Cotação_USD':'first',
    'Margem de lucro':'first',
    'Frete preço':'first',
    'Data':'first',
    'estado':'first',
    'cidade':'first',
    'Frete':'first' # Assumindo que a coluna frete é igual para pedidos únicos
}


In [119]:
# Agrupar por 'Pedido' e aplicar a função personalizada para 'Estoque', somando as colunas numéricas

df_agrupado = df_final.groupby('Pedido', as_index=True).agg({
    'Estoque': determinar_estoque,**operacoes}
)

df_agrupado = df_agrupado.reset_index()

In [120]:
# Renomear a coluna 'Estoque' para 'Tipo de pedido'

df_agrupado = df_agrupado.rename(columns={'Estoque': 'Tipo de pedido'})

In [121]:
df_final['Data'] = pd.to_datetime(df_final['Data'],dayfirst=True)
df_final['Data_TD'] = pd.to_datetime(df_final['Data_TD'],dayfirst=True)
df_agrupado['Data'] = pd.to_datetime(df_agrupado['Data'],dayfirst=True)


In [122]:
colunas_numericas = ['Faturamento líquido','Custo_Unitario_USD','Custo_Unitario_BRL','Lucro bruto', 'Dias_entre_datas','Cotação_USD','Margem de lucro']
df_final[colunas_numericas] = df_final[colunas_numericas].apply(pd.to_numeric)

print(df_final.dtypes)

PedidoSKU                      object
Pedido                         object
Data                   datetime64[ns]
Produto                        object
Cor                            object
Tamanho                        object
SKU                            object
Quantidade                      int64
Faturamento bruto              object
Descontos                      object
Faturamento líquido           float64
Frete                          object
Frete preço                    object
cidade                         object
estado                         object
Estoque                        object
Data_TD                datetime64[ns]
Status                         object
T&D                            object
Motivo_T&D                     object
Comentário                     object
Dias_entre_datas              float64
Cotação_USD                   float64
Custo_Unitario_USD            float64
Custo_Unitario_BRL            float64
Custo_BRL                     float64
Lucro bruto 

In [123]:
df_final['Data'] = df_final['Data'].dt.strftime('%d/%m/%Y')
df_final['Data_TD'] = df_final['Data_TD'].dt.strftime('%d/%m/%Y')
df_agrupado['Data'] = df_agrupado['Data'].dt.strftime('%d/%m/%Y')

df_final.to_csv(r"C:\Users\GabrielCielo\Desktop\consolatio\Databases\Main.csv",index=False,sep=';',decimal='.')
df_agrupado.to_csv(r"C:\Users\GabrielCielo\Desktop\consolatio\Databases\OrdersDB.csv",index=False,sep=';',decimal='.')

In [124]:
pedidos_acima_de_400 = df_agrupado[df_agrupado['Faturamento líquido'] > 400].shape[0]

print(f"Há um total de {pedidos_acima_de_400} pedidos acima de R$400,00 e que, portanto, receberam frete grátis")

proporção_acima_de_400 = (df_agrupado[df_agrupado['Faturamento líquido'] > 400].shape[0] / df_agrupado.shape[0])*100

print(f"O que representa {proporção_acima_de_400}% de todos os pedidos")

Há um total de 2936 pedidos acima de R$400,00 e que, portanto, receberam frete grátis
O que representa 31.370872956512446% de todos os pedidos


In [125]:
pedidos_acima_de_850 = df_agrupado[df_agrupado['Faturamento líquido'] > 850].shape[0]

print(f"Há um total de {pedidos_acima_de_850} pedidos acima de R$850,00.")

proporção_acima_de_850 = (df_agrupado[df_agrupado['Faturamento líquido'] > 850].shape[0] / df_agrupado.shape[0])*100

print(f"O que representa {proporção_acima_de_850}% de todos os pedidos")

Há um total de 618 pedidos acima de R$850,00.
O que representa 6.603269580083341% de todos os pedidos


In [126]:
df_x = df_final[df_final['Produto'] == 'Camisa X-Tretch']
df_x['Lucro bruto'].sum()

192360.00947400002

In [127]:
df_final.groupby('Produto').agg({'Lucro bruto':'sum'})

,Lucro bruto
Produto,
Calça Neo,6554.790000
Calça X-Tretch,97688.072148
Camisa Polo Ultra,185708.662314
Camisa Ultra-Stretch,289806.212543
Camisa X-Tretch,192360.009474
Camisa X-Tretch Manga Curta,1004.502900
Camiseta TecModal,29498.030000
Kit 10 Cuecas,1524.402850
Kit 5 Cuecas,1560.274500
